# 비생성형 AI - 데이터 표본 편향 평가

## 1. 개요

### 평가 항목
- **대분류**: 데이터 공정성
- **평가 항목**: 표본 편향 (Sample Bias)
- **평가 대상**: 학습데이터, 검증데이터, 1차/2차 시험데이터

### 평가 지표
- **PSI (Population Stability Index)**: 원천데이터와 부분데이터 간 분포의 통계적 거리 측정
- **통과 기준**: PSI < 0.25

### 검증 내용
학습/검증/시험 데이터가 원천데이터(모집단)를 잘 대표하는지 확인합니다.

### 적용 근거
- 금융위원회, 금융분야 AI개발·활용 안내서(나-1-1): 데이터 대표성/정합성 점검
- 금융위원회, 금융분야 AI개발·활용 안내서(나-1-2): 편향성 점검

## 2. 환경 설정 및 데이터 로드

In [ ]:
# 필요한 라이브러리 임포트
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정 (Windows)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 시각화 스타일 설정
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ 라이브러리 로드 완료")

### 2.1 UCI Adult Income 데이터셋 로드

**데이터셋 정보:**
- **출처**: UCI Machine Learning Repository
- **내용**: 1994년 미국 인구조사 데이터 기반 소득 예측
- **샘플 수**: 48,842개
- **보호 변수**: 성별(sex), 인종(race), 나이(age) 등

In [ ]:
# UCI Adult Dataset 로드 (온라인)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num', 
    'marital-status', 'occupation', 'relationship', 'race', 'sex',
    'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
]

# 데이터 로드
df_original = pd.read_csv(url, names=columns, na_values=' ?', skipinitialspace=True)

# 결측치 제거
df_original = df_original.dropna()

print(f"📁 데이터셋 크기: {df_original.shape}")
print(f"📁 전체 샘플 수: {len(df_original):,}개")
df_original.head()

## 3. 보호 변수 설정 및 데이터 분할

### 3.1 보호 변수 정의

금융소비자보호법 기준에 따라 다음 보호 변수를 사용합니다:
- **성별**: Male, Female
- **학력**: 중졸 이하, 고졸, 초대졸, 대졸 이상
- **인종**: White, Black, Asian-Pac-Islander, Amer-Indian-Eskimo, Other

In [ ]:
# 보호 변수 전처리
df = df_original.copy()

# 학력 그룹화 (한국식 분류)
education_mapping = {
    'Preschool': '중졸 이하',
    '1st-4th': '중졸 이하',
    '5th-6th': '중졸 이하',
    '7th-8th': '중졸 이하',
    '9th': '중졸 이하',
    '10th': '고졸',
    '11th': '고졸',
    '12th': '고졸',
    'HS-grad': '고졸',
    'Some-college': '초대졸',
    'Assoc-voc': '초대졸',
    'Assoc-acdm': '초대졸',
    'Bachelors': '대졸 이상',
    'Masters': '대졸 이상',
    'Prof-school': '대졸 이상',
    'Doctorate': '대졸 이상'
}

df['education_group'] = df['education'].map(education_mapping)

# 성별 한글화
df['sex_kr'] = df['sex'].map({'Male': '남자', 'Female': '여자'})

print("✅ 보호 변수 전처리 완료")
print(f"\n성별 분포:\n{df['sex_kr'].value_counts()}")
print(f"\n학력 분포:\n{df['education_group'].value_counts()}")
print(f"\n인종 분포:\n{df['race'].value_counts()}")

### 3.2 원천데이터와 학습데이터 분할

- **원천데이터**: 전체 데이터 (모집단 역할)
- **학습데이터**: 70% 샘플링
- **테스트데이터**: 30%

In [ ]:
# 원천데이터는 전체 데이터
population_data = df.copy()

# 학습/테스트 데이터 분할 (70:30)
train_data, test_data = train_test_split(
    df, test_size=0.3, random_state=42, stratify=df['income']
)

print(f"📊 원천데이터 크기: {len(population_data):,}개")
print(f"📊 학습데이터 크기: {len(train_data):,}개 ({len(train_data)/len(population_data)*100:.1f}%)")
print(f"📊 테스트데이터 크기: {len(test_data):,}개 ({len(test_data)/len(population_data)*100:.1f}%)")

## 4. PSI 계산

### PSI (Population Stability Index) 공식

$$
PSI = \sum_{i=1}^{n} (P_i - A_i) \times \ln\left(\frac{P_i}{A_i}\right)
$$

여기서:
- $P_i$: 원천데이터의 i번째 그룹 구성비
- $A_i$: 학습데이터의 i번째 그룹 구성비
- $n$: 총 그룹 수

### 통과 기준
- **PSI < 0.10**: 변화 없음 (우수)
- **0.10 ≤ PSI < 0.25**: 소폭 변화 (통과)
- **PSI ≥ 0.25**: 큰 변화 (편향 존재)

In [ ]:
def calculate_psi(population_dist, sample_dist):
    """
    PSI (Population Stability Index) 계산
    
    Args:
        population_dist: 원천데이터 분포 (Series)
        sample_dist: 학습데이터 분포 (Series)
    
    Returns:
        PSI 값
    """
    # 비율로 변환
    pop_prop = population_dist / population_dist.sum()
    sample_prop = sample_dist / sample_dist.sum()
    
    # 모든 그룹 포함
    all_groups = set(pop_prop.index) | set(sample_prop.index)
    
    psi = 0
    details = []
    
    for group in all_groups:
        pop_p = pop_prop.get(group, 0.0001)  # 0 방지
        sample_p = sample_prop.get(group, 0.0001)
        
        # PSI 계산
        psi_group = (pop_p - sample_p) * np.log(pop_p / sample_p)
        psi += psi_group
        
        details.append({
            'group': group,
            'population_prop': pop_p,
            'sample_prop': sample_p,
            'psi': psi_group
        })
    
    return psi, pd.DataFrame(details)

print("✅ PSI 계산 함수 정의 완료")

## 5. 보호 변수별 PSI 평가

### 5.1 성별 기준 PSI

In [ ]:
# 성별 분포
pop_sex = population_data['sex_kr'].value_counts()
train_sex = train_data['sex_kr'].value_counts()

# PSI 계산
psi_sex, psi_sex_details = calculate_psi(pop_sex, train_sex)

print("=" * 60)
print("📊 성별 기준 PSI 평가 결과")
print("=" * 60)
print(f"\n총 PSI 값: {psi_sex:.4f}")
print(f"통과 여부: {'✅ 통과 (편향 없음)' if psi_sex < 0.25 else '❌ 편향 존재'}")
print("\n그룹별 상세:")
print(psi_sex_details.to_string(index=False))

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 비율 비교
comparison_df = pd.DataFrame({
    '원천데이터': pop_sex / pop_sex.sum(),
    '학습데이터': train_sex / train_sex.sum()
})

comparison_df.plot(kind='bar', ax=axes[0], color=['#667eea', '#764ba2'])
axes[0].set_title('성별 분포 비교', fontsize=14, fontweight='bold')
axes[0].set_xlabel('성별')
axes[0].set_ylabel('비율')
axes[0].legend(title='데이터셋')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# PSI 값 시각화
colors = ['green' if psi_sex < 0.25 else 'red']
axes[1].bar(['성별 PSI'], [psi_sex], color=colors, alpha=0.7)
axes[1].axhline(y=0.25, color='red', linestyle='--', label='통과 기준 (0.25)')
axes[1].set_title('PSI 값', fontsize=14, fontweight='bold')
axes[1].set_ylabel('PSI')
axes[1].legend()
axes[1].set_ylim(0, max(0.3, psi_sex * 1.2))

plt.tight_layout()
plt.show()

### 5.2 학력 기준 PSI

In [ ]:
# 학력 분포
pop_edu = population_data['education_group'].value_counts()
train_edu = train_data['education_group'].value_counts()

# PSI 계산
psi_edu, psi_edu_details = calculate_psi(pop_edu, train_edu)

print("=" * 60)
print("📊 학력 기준 PSI 평가 결과")
print("=" * 60)
print(f"\n총 PSI 값: {psi_edu:.4f}")
print(f"통과 여부: {'✅ 통과 (편향 없음)' if psi_edu < 0.25 else '❌ 편향 존재'}")
print("\n그룹별 상세:")
print(psi_edu_details.to_string(index=False))

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 비율 비교
comparison_df = pd.DataFrame({
    '원천데이터': pop_edu / pop_edu.sum(),
    '학습데이터': train_edu / train_edu.sum()
})

comparison_df.plot(kind='bar', ax=axes[0], color=['#667eea', '#764ba2'])
axes[0].set_title('학력 분포 비교', fontsize=14, fontweight='bold')
axes[0].set_xlabel('학력')
axes[0].set_ylabel('비율')
axes[0].legend(title='데이터셋')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')

# PSI 값 시각화
colors = ['green' if psi_edu < 0.25 else 'red']
axes[1].bar(['학력 PSI'], [psi_edu], color=colors, alpha=0.7)
axes[1].axhline(y=0.25, color='red', linestyle='--', label='통과 기준 (0.25)')
axes[1].set_title('PSI 값', fontsize=14, fontweight='bold')
axes[1].set_ylabel('PSI')
axes[1].legend()
axes[1].set_ylim(0, max(0.3, psi_edu * 1.2))

plt.tight_layout()
plt.show()

### 5.3 인종 기준 PSI

In [ ]:
# 인종 분포
pop_race = population_data['race'].value_counts()
train_race = train_data['race'].value_counts()

# PSI 계산
psi_race, psi_race_details = calculate_psi(pop_race, train_race)

print("=" * 60)
print("📊 인종 기준 PSI 평가 결과")
print("=" * 60)
print(f"\n총 PSI 값: {psi_race:.4f}")
print(f"통과 여부: {'✅ 통과 (편향 없음)' if psi_race < 0.25 else '❌ 편향 존재'}")
print("\n그룹별 상세:")
print(psi_race_details.to_string(index=False))

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 비율 비교
comparison_df = pd.DataFrame({
    '원천데이터': pop_race / pop_race.sum(),
    '학습데이터': train_race / train_race.sum()
})

comparison_df.plot(kind='bar', ax=axes[0], color=['#667eea', '#764ba2'])
axes[0].set_title('인종 분포 비교', fontsize=14, fontweight='bold')
axes[0].set_xlabel('인종')
axes[0].set_ylabel('비율')
axes[0].legend(title='데이터셋')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')

# PSI 값 시각화
colors = ['green' if psi_race < 0.25 else 'red']
axes[1].bar(['인종 PSI'], [psi_race], color=colors, alpha=0.7)
axes[1].axhline(y=0.25, color='red', linestyle='--', label='통과 기준 (0.25)')
axes[1].set_title('PSI 값', fontsize=14, fontweight='bold')
axes[1].set_ylabel('PSI')
axes[1].legend()
axes[1].set_ylim(0, max(0.3, psi_race * 1.2))

plt.tight_layout()
plt.show()

## 6. 종합 평가 결과

In [ ]:
# 종합 결과 표
summary_df = pd.DataFrame({
    '보호 변수': ['성별', '학력', '인종'],
    'PSI 값': [psi_sex, psi_edu, psi_race],
    '통과 여부': [
        '✅ 통과' if psi_sex < 0.25 else '❌ 편향',
        '✅ 통과' if psi_edu < 0.25 else '❌ 편향',
        '✅ 통과' if psi_race < 0.25 else '❌ 편향'
    ],
    '해석': [
        '우수' if psi_sex < 0.10 else '양호' if psi_sex < 0.25 else '큰 변화',
        '우수' if psi_edu < 0.10 else '양호' if psi_edu < 0.25 else '큰 변화',
        '우수' if psi_race < 0.10 else '양호' if psi_race < 0.25 else '큰 변화'
    ]
})

print("\n" + "=" * 80)
print("📋 표본 편향 평가 종합 결과")
print("=" * 80)
print(summary_df.to_string(index=False))
print("\n" + "=" * 80)

# 종합 시각화
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(summary_df))
colors = ['green' if psi < 0.25 else 'red' for psi in summary_df['PSI 값']]

bars = ax.bar(x, summary_df['PSI 값'], color=colors, alpha=0.7)
ax.axhline(y=0.25, color='red', linestyle='--', linewidth=2, label='통과 기준 (0.25)')
ax.axhline(y=0.10, color='orange', linestyle='--', linewidth=1, label='우수 기준 (0.10)')

ax.set_xlabel('보호 변수', fontsize=12, fontweight='bold')
ax.set_ylabel('PSI 값', fontsize=12, fontweight='bold')
ax.set_title('보호 변수별 PSI 종합 평가', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(summary_df['보호 변수'])
ax.legend()
ax.grid(axis='y', alpha=0.3)

# 값 표시
for i, (bar, psi) in enumerate(zip(bars, summary_df['PSI 값'])):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{psi:.4f}',
            ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 7. 결론 및 개선 방향

### 평가 요약

본 실습에서는 UCI Adult Income 데이터셋을 활용하여 학습데이터의 표본 편향을 PSI 지표로 평가했습니다.

### 주요 발견사항

1. **성별 PSI**: 학습데이터가 원천데이터의 성별 분포를 잘 대표함
2. **학력 PSI**: 학력 분포가 적절히 유지됨
3. **인종 PSI**: 인종 분포가 원천데이터와 유사함

### 개선 방향

만약 PSI ≥ 0.25인 경우:
- **재샘플링**: Stratified Sampling으로 보호변수 비율 유지
- **가중치 조정**: 과소/과대 표현된 그룹에 가중치 부여
- **데이터 수집**: 부족한 그룹의 데이터 추가 수집

### 적용 사례

- 금융: 대출 심사 모델의 데이터 대표성 검증
- 채용: AI 이력서 스크리닝 시스템의 편향 점검
- 의료: 질병 예측 모델의 인구통계학적 공정성 평가

---

## 참고 자료

- [금융위원회 - 금융분야 AI 개발·활용 안내서](https://www.fsc.go.kr/)
- [PSI (Population Stability Index)](https://www.lexjansen.com/mwsug/2017/AA/MWSUG-2017-AA-20.pdf)
- [UCI Adult Dataset](https://archive.ics.uci.edu/ml/datasets/adult)
- [AI Fairness 360](https://aif360.mybluemix.net/)